# Nova — توليد صور AI وإنتاج فيديو وثائقي كامل مجاناً عبر Kaggle GPU

هذا الدفتر يولّد 15 صورة أصلية باستخدام **Tongyi-MAI/Z-Image-Turbo** (بدون Pollinations وبدون FLUX)، ثم يشغّل سكربت Nova لإنتاج فيديو MP4 بدقة 1080p مع تعليق إنجليزي وترجمة متزامنة.

**التشغيل:** افتح إعدادات Kaggle واختر Accelerator = GPU وInternet = On، ثم Run All. الحصة المجانية وتوفر GPU يخضعان لقيود Kaggle. لا يحتاج مفتاح API. الفيديو النهائي يظهر في آخر خلية ويمكن تنزيله من رابط الملف.

In [ ]:
!git clone --depth 1 https://github.com/nova-api-del/nova.git /kaggle/working/nova
%cd /kaggle/working/nova
!pip -q install -U git+https://github.com/huggingface/diffusers transformers accelerate sentencepiece protobuf edge-tts requests pillow
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-dejavu-core

In [ ]:
import os, sys, json, zipfile, torch
from pathlib import Path
from diffusers import ZImagePipeline
sys.path.insert(0, '/kaggle/working/nova')
from scripts.longform_image_video import SCENES

assert torch.cuda.is_available(), 'Turn on a GPU in Kaggle Notebook settings, then restart the session.'
props = torch.cuda.get_device_properties(0)
major, minor = torch.cuda.get_device_capability(0)
dtype = torch.bfloat16 if major >= 8 else torch.float16
print('GPU:', props.name, '| VRAM:', round(props.total_memory/1024**3, 1), 'GB | dtype:', dtype)

OUT = Path('/kaggle/working/nova_ai_frames')
OUT.mkdir(parents=True, exist_ok=True)
pipe = ZImagePipeline.from_pretrained(
    'Tongyi-MAI/Z-Image-Turbo',
    torch_dtype=dtype,
    low_cpu_mem_usage=False,
)
pipe.enable_model_cpu_offload()
print('Z-Image Turbo loaded. Starting 15 scenes...')

In [ ]:
records=[]
for i, (title, paragraph, visual) in enumerate(SCENES, 1):
    prompt = (
      'Photorealistic premium natural-history documentary still, cinematic IMAX underwater photography, '
      'wide 16:9 composition, extraordinary detail, physically realistic marine biology, dramatic but natural '
      'volumetric lighting, rich deep-ocean blues, believable scale, professional BBC Earth cinematography. '
      'No text, no labels, no watermark. Scene: ' + visual + '.'
    )
    seed = 26000 + i
    image = pipe(
      prompt=prompt, height=576, width=1024, num_inference_steps=9, guidance_scale=0.0,
      generator=torch.Generator(device='cpu').manual_seed(seed)
    ).images[0]
    path = OUT / f'scene_{i:02d}.png'
    image.save(path)
    records.append({'scene':i,'title':title,'model':'Tongyi-MAI/Z-Image-Turbo','seed':seed,'prompt':prompt,'file':path.name})
    print(f'[{i:02d}/15] Generated: {title}')
    del image
    if torch.cuda.is_available(): torch.cuda.empty_cache()

(OUT/'manifest.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
zip_path='/kaggle/working/nova_ai_frames.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()): z.write(p, arcname=p.name)
print('READY:', zip_path, '| images:', len(records))

In [ ]:
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/nova_ai_frames.zip'))

In [ ]:
# 5) Render the complete five-minute documentary using AI frames only
%cd /kaggle/working/nova
import os
os.environ['AI_FRAMES_DIR'] = '/kaggle/working/nova_ai_frames'
os.environ['REQUIRE_AI_IMAGES'] = 'true'
os.environ['TTS_VOICE'] = 'en-US-GuyNeural'
!python scripts/longform_image_video.py


In [ ]:
# 6) Preview and expose the final MP4 download link
from IPython.display import Video, FileLink, display
video_path = '/kaggle/working/nova/output/longform/nova_five_minute_documentary.mp4'
assert Path(video_path).exists(), 'Video render failed. Check the previous cell output.'
display(Video(video_path, embed=False, width=800))
display(FileLink(video_path))
print('AI image manifest:', '/kaggle/working/nova/output/longform/metadata.json')
